In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from nemosis import dynamic_data_compiler
import os

os.makedirs('./data/nemosis_cache', exist_ok=True)
CACHE = './data/nemosis_cache'

# Pull dispatch load data — this shows actual MW output per generator per 5-min interval
# INTERVENTION=0 filters out intervention periods (same reason as in price data)
dispatch = dynamic_data_compiler(
    '2025/01/01 00:00:00',
    '2026/02/01 00:00:00',
    'DISPATCHLOAD', CACHE,
    fformat='parquet',
    filter_cols=['INTERVENTION'],
    filter_values=([0],)
)

# Keep only essential columns
dispatch = dispatch[['SETTLEMENTDATE', 'DUID', 'DISPATCHLOAD']].copy()

# Align time convention with Price Explorer — interval start, not end
dispatch['SETTLEMENTDATE'] = dispatch['SETTLEMENTDATE'] - pd.Timedelta('5min')

print(f"Dispatch data shape: {dispatch.shape}")
print(f"Memory usage: {dispatch.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Date range: {dispatch['SETTLEMENTDATE'].min()} to {dispatch['SETTLEMENTDATE'].max()}")
print(f"\nNumber of unique generators (DUIDs): {dispatch['DUID'].nunique()}")
print("\nSample:")
print(dispatch.head(20))

INFO: Compiling data for table DISPATCHLOAD
INFO: Downloading data for table DISPATCHLOAD, year 2024, month 12
INFO: Creating parquet file for DISPATCHLOAD, 2024, 12
INFO: Downloading data for table DISPATCHLOAD, year 2025, month 01
INFO: Creating parquet file for DISPATCHLOAD, 2025, 01
INFO: Downloading data for table DISPATCHLOAD, year 2025, month 02
INFO: Creating parquet file for DISPATCHLOAD, 2025, 02
INFO: Downloading data for table DISPATCHLOAD, year 2025, month 03
INFO: Creating parquet file for DISPATCHLOAD, 2025, 03
INFO: Downloading data for table DISPATCHLOAD, year 2025, month 04
INFO: Creating parquet file for DISPATCHLOAD, 2025, 04
INFO: Downloading data for table DISPATCHLOAD, year 2025, month 05
INFO: Creating parquet file for DISPATCHLOAD, 2025, 05
INFO: Downloading data for table DISPATCHLOAD, year 2025, month 06


In [ ]:
# Pull AEMO Registration data — this maps each generator to its fuel type,
# capacity, region, and other attributes
# This is the static reference data that tells us WHAT each DUID actually is

registration = dynamic_data_compiler(
    '2025/01/01 00:00:00',
    '2026/01/01 00:00:00',
    'DUDETAILSUMMARY', CACHE,
    fformat='parquet'
)

# Keep relevant fields — we need fuel type, capacity, region, and station name
registration = registration[[
    'DUID', 'FUEL_SOURCE_PRIMARY', 'REG_CAPACITY', 'REGIONID', 'STATIONNAME'
]].copy()

# DUDETAILSUMMARY can have multiple rows per DUID (e.g. if capacity changed)
# Take the most recent entry per DUID
registration = registration.drop_duplicates(subset='DUID', keep='last')

print(f"Registration records: {len(registration)}")
print(f"\nFuel types in NEM:")
print(registration['FUEL_SOURCE_PRIMARY'].value_counts())
print("\nSample:")
print(registration.head(20))

In [ ]:
# Join dispatch data to registration to get fuel type for each generator
# This is the core dataset for all subsequent analysis

dispatch_enriched = dispatch.merge(
    registration[['DUID', 'FUEL_SOURCE_PRIMARY', 'REG_CAPACITY', 'REGIONID']],
    on='DUID',
    how='left'
)

# Check for unmapped DUIDs — these are either missing from registration or are
# non-generator units (loads, network service providers)
unmapped = dispatch_enriched['FUEL_SOURCE_PRIMARY'].isna().sum()
print(f"Unmapped dispatch records: {unmapped:,} ({100*unmapped/len(dispatch_enriched):.2f}%)")

if unmapped > 0:
    print("\nTop unmapped DUIDs by dispatch volume:")
    unmapped_duids = dispatch_enriched[dispatch_enriched['FUEL_SOURCE_PRIMARY'].isna()]
    top_unmapped = (unmapped_duids
        .groupby('DUID')['DISPATCHLOAD']
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    print(top_unmapped)

# Drop unmapped records for clean analysis
dispatch_enriched = dispatch_enriched.dropna(subset=['FUEL_SOURCE_PRIMARY'])

print(f"\nEnriched dispatch data shape: {dispatch_enriched.shape}")
print(dispatch_enriched.head())

In [ ]:
# Create a simplified fuel type mapping for cleaner visualisation
# AEMO's fuel type codes are granular — we'll group into major categories

fuel_map = {
    # Coal
    'Brown Coal': 'Coal',
    'Black Coal': 'Coal',
    'Coal Seam Methane': 'Coal',
    
    # Gas
    'Natural Gas': 'Gas',
    'Natural Gas / Fuel Oil': 'Gas',
    'Diesel': 'Gas',  # Group liquid fuels with gas for peaking
    'Kerosene': 'Gas',
    
    # Hydro
    'Hydro': 'Hydro',
    
    # Renewables
    'Wind': 'Wind',
    'Solar': 'Solar',
    'Biomass': 'Biomass',
    'Bagasse': 'Biomass',
    
    # Battery
    'Battery': 'Battery',
    'Battery / Solar': 'Battery',  # Hybrid systems
}

dispatch_enriched['FUEL_TYPE'] = (
    dispatch_enriched['FUEL_SOURCE_PRIMARY']
    .map(fuel_map)
    .fillna('Other')
)

print("Fuel type distribution (MW-hours):")
fuel_energy = dispatch_enriched.groupby('FUEL_TYPE')['DISPATCHLOAD'].sum() / 12  # 5min → hourly
print(fuel_energy.sort_values(ascending=False))
print(f"\nTotal energy: {fuel_energy.sum()/1e6:.1f} TWh")

In [ ]:
# Merit Order Stack — VIC, single day (2025-07-15)
# Shows the dispatch stack by fuel type across 24 hours
# Coal at the bottom (baseload), gas in the middle, peakers at the top
# This is the visual representation of the merit order — cheap generators
# run continuously, expensive generators only during peaks

target_date = '2025-07-15'
region = 'VIC1'

# Filter to target day and region, resample to hourly
day_data = dispatch_enriched[
    (dispatch_enriched['SETTLEMENTDATE'].dt.date == pd.to_datetime(target_date).date()) &
    (dispatch_enriched['REGIONID'] == region)
].copy()

# Aggregate to hourly by fuel type
day_data['hour'] = day_data['SETTLEMENTDATE'].dt.hour
stack = day_data.groupby(['hour', 'FUEL_TYPE'])['DISPATCHLOAD'].sum().unstack(fill_value=0)

# Define stack order (baseload → peaking) and colours
stack_order = ['Coal', 'Hydro', 'Wind', 'Solar', 'Gas', 'Battery', 'Biomass', 'Other']
stack_colors = {
    'Coal': '#4A4A4A',
    'Gas': '#FF6B6B',
    'Hydro': '#4ECDC4',
    'Wind': '#95E1D3',
    'Solar': '#FFD93D',
    'Battery': '#9D65C9',
    'Biomass': '#6BCB77',
    'Other': '#B8B8B8'
}

# Only plot fuel types present in the data
stack_order_present = [f for f in stack_order if f in stack.columns]

fig, ax = plt.subplots(figsize=(14, 6))
stack[stack_order_present].plot.area(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    alpha=0.9,
    linewidth=0
)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Dispatch (MW)')
ax.set_title(f'{region} Merit Order Stack — {target_date}')
ax.legend(title='Fuel Type', loc='upper left')
ax.set_xticks(range(24))
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nPeak demand: {stack[stack_order_present].sum(axis=1).max():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmax()}")
print(f"Min demand: {stack[stack_order_present].sum(axis=1).min():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmin()}")

In [ ]:
# Merit Order Stack — SA, single day (same date for comparison)
# SA has a fundamentally different generation mix to VIC:
#   - No coal (last coal plant closed 2016)
#   - High wind penetration
#   - Gas for firming and peaking
#   - Hornsdale battery for frequency response and arbitrage
# The stack shape will be visually distinct — wind dominates during windy hours,
# gas fills gaps, battery charges/discharges around price peaks

region = 'SA1'

day_data = dispatch_enriched[
    (dispatch_enriched['SETTLEMENTDATE'].dt.date == pd.to_datetime(target_date).date()) &
    (dispatch_enriched['REGIONID'] == region)
].copy()

day_data['hour'] = day_data['SETTLEMENTDATE'].dt.hour
stack = day_data.groupby(['hour', 'FUEL_TYPE'])['DISPATCHLOAD'].sum().unstack(fill_value=0)

stack_order_present = [f for f in stack_order if f in stack.columns]

fig, ax = plt.subplots(figsize=(14, 6))
stack[stack_order_present].plot.area(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    alpha=0.9,
    linewidth=0
)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Dispatch (MW)')
ax.set_title(f'{region} Merit Order Stack — {target_date}')
ax.legend(title='Fuel Type', loc='upper left')
ax.set_xticks(range(24))
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nPeak demand: {stack[stack_order_present].sum(axis=1).max():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmax()}")
print(f"Min demand: {stack[stack_order_present].sum(axis=1).min():.0f} MW at hour {stack[stack_order_present].sum(axis=1).idxmin()}")

# Battery behaviour check — charging shows as negative dispatch
if 'Battery' in stack.columns:
    battery_charge_hours = (stack['Battery'] < 0).sum()
    print(f"\nBattery charged for {battery_charge_hours} hours (negative dispatch)")
    print(f"Battery discharged for {(stack['Battery'] > 0).sum()} hours (positive dispatch)")

In [ ]:
# Marginal Generator Identification — VIC, single day
# The marginal generator is the unit whose short-run marginal cost (SRMC) sets
# the spot price in each interval. In practice, this is the most expensive unit
# dispatched in that interval.
#
# We don't have bid data here (that's BIDDAYOFFER, much heavier dataset), so we
# use a proxy: for each interval, identify which fuel type is at the "top" of the
# stack — i.e. the most expensive fuel type running at that time.
#
# Simplified SRMC hierarchy (actual bids vary, but this is the conceptual order):
#   Coal < Hydro < Wind/Solar (≈0) < Gas CCGT < Gas OCGT < Diesel
#
# This chart shows which fuel type was "marginal" in each hour — when you see
# Gas dominating, it means gas generators were setting price.

region = 'VIC1'

day_data = dispatch_enriched[
    (dispatch_enriched['SETTLEMENTDATE'].dt.date == pd.to_datetime(target_date).date()) &
    (dispatch_enriched['REGIONID'] == region)
].copy()

# For each interval, find the "highest cost" fuel type with non-zero dispatch
# Rank fuel types by typical SRMC (higher = more expensive)
srmc_rank = {
    'Wind': 1,
    'Solar': 1,
    'Hydro': 2,
    'Coal': 3,
    'Biomass': 4,
    'Gas': 5,
    'Battery': 6,  # Battery marginal cost is opportunity cost, typically high
    'Other': 7
}

day_data['SRMC_RANK'] = day_data['FUEL_TYPE'].map(srmc_rank)

# For each interval, find the fuel type with highest SRMC_RANK that has dispatch > 0
# Group by 5-min interval, filter to dispatched units, take max rank
marginal = (day_data[day_data['DISPATCHLOAD'] > 0]
    .groupby('SETTLEMENTDATE')
    .apply(lambda g: g.loc[g['SRMC_RANK'].idxmax(), 'FUEL_TYPE'])
    .reset_index()
    .rename(columns={0: 'MARGINAL_FUEL'})
)

marginal['hour'] = marginal['SETTLEMENTDATE'].dt.hour

# Count how many intervals each fuel type was marginal in each hour
marginal_counts = (marginal
    .groupby(['hour', 'MARGINAL_FUEL'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 6))
marginal_counts.plot.bar(
    ax=ax,
    stacked=True,
    color=[stack_colors.get(f, '#B8B8B8') for f in marginal_counts.columns],
    width=0.8
)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Number of 5-min intervals')
ax.set_title(f'{region} Marginal Fuel Type — {target_date}\n'
             f'(Proxy: highest-SRMC fuel dispatched in each interval)')
ax.legend(title='Marginal Fuel', loc='upper left')
ax.set_xticklabels(range(24), rotation=0)
plt.tight_layout()
plt.show()

print("\nMarginal fuel summary for the day:")
print(marginal['MARGINAL_FUEL'].value_counts())

In [ ]:
# Capacity Factor Analysis — Actual dispatch vs. registered capacity
# Capacity factor = actual output / nameplate capacity
# For each generator, calculate average capacity factor over the full year
# This reveals:
#   - Baseload units (coal, nuclear if it existed) → CF near 80–90%
#   - Wind → CF around 30–35% (varies by site)
#   - Solar → CF around 20–25% (night = 0, clouds, seasonal tilt)
#   - Peakers (OCGT) → CF under 5% (only run during price spikes)
#   - Outages and de-ratings show up as unexpectedly low CF for baseload units

# Calculate total energy output per DUID over the year
energy_by_duid = dispatch_enriched.groupby('DUID').agg({
    'DISPATCHLOAD': 'sum',
    'REG_CAPACITY': 'first',
    'FUEL_TYPE': 'first',
    'REGIONID': 'first'
}).reset_index()

# Total possible energy = capacity × number of intervals
# 2025 is not a leap year → 365 days × 24 hours × 12 intervals/hour
total_intervals = 365 * 24 * 12
energy_by_duid['MAX_ENERGY'] = energy_by_duid['REG_CAPACITY'] * total_intervals
energy_by_duid['CAPACITY_FACTOR'] = (
    energy_by_duid['DISPATCHLOAD'] / energy_by_duid['MAX_ENERGY']
).clip(upper=1.0)  # Cap at 100% in case of data errors

# Remove generators with zero capacity or zero output (not commissioned, retired)
energy_by_duid = energy_by_duid[
    (energy_by_duid['REG_CAPACITY'] > 0) &
    (energy_by_duid['DISPATCHLOAD'] > 0)
]

print(f"Generators with non-zero output: {len(energy_by_duid)}")
print("\nCapacity factor by fuel type (median):")
print(energy_by_duid.groupby('FUEL_TYPE')['CAPACITY_FACTOR'].median().sort_values(ascending=False))

# Scatter plot: capacity factor vs. registered capacity, coloured by fuel type
fig, ax = plt.subplots(figsize=(12, 7))

for fuel in ['Coal', 'Gas', 'Hydro', 'Wind', 'Solar', 'Battery']:
    subset = energy_by_duid[energy_by_duid['FUEL_TYPE'] == fuel]
    if len(subset) > 0:
        ax.scatter(
            subset['REG_CAPACITY'],
            subset['CAPACITY_FACTOR'] * 100,
            label=fuel,
            color=stack_colors.get(fuel, '#B8B8B8'),
            alpha=0.6,
            s=50
        )

ax.set_xlabel('Registered Capacity (MW)')
ax.set_ylabel('Capacity Factor (%)')
ax.set_title('Generator Capacity Factor vs. Nameplate Capacity — 2025')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, None)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# Identify Low-CF Baseload Units — Potential Outages
# Coal units should run at CF > 70% under normal conditions
# Anything below 50% suggests extended outage or de-rating
# This is the kind of analysis you'd do when investigating why prices spiked
# in a particular period — "was Loy Yang A offline?"

coal_units = energy_by_duid[energy_by_duid['FUEL_TYPE'] == 'Coal'].copy()
coal_units = coal_units.sort_values('CAPACITY_FACTOR')

print("Coal units with capacity factor < 50%:")
low_cf_coal = coal_units[coal_units['CAPACITY_FACTOR'] < 0.5]
print(low_cf_coal[['DUID', 'REG_CAPACITY', 'CAPACITY_FACTOR', 'REGIONID']])

if len(low_cf_coal) > 0:
    print("\nThese units may have experienced extended outages in 2025.")
    print("Cross-reference with AEMO outage notices or DISPATCHLOAD time series.")
else:
    print("\nAll coal units operated at normal capacity factors — no major outages detected.")

In [ ]:
# Wind and Solar Capacity Factor Distribution
# Wind CF varies by site (coastal vs. inland, hub height, turbine model)
# Solar CF varies by latitude, panel tilt, and tracking vs. fixed
# The distribution of CF within each fuel type tells you about fleet diversity

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wind
wind_cf = energy_by_duid[energy_by_duid['FUEL_TYPE'] == 'Wind']['CAPACITY_FACTOR'] * 100
axes[0].hist(wind_cf, bins=30, color=stack_colors['Wind'], edgecolor='black', alpha=0.7)
axes[0].axvline(wind_cf.median(), color='red', linestyle='--', linewidth=2, label=f'Median: {wind_cf.median():.1f}%')
axes[0].set_xlabel('Capacity Factor (%)')
axes[0].set_ylabel('Number of wind farms')
axes[0].set_title('Wind Farm Capacity Factor Distribution — 2025')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Solar
solar_cf = energy_by_duid[energy_by_duid['FUEL_TYPE'] == 'Solar']['CAPACITY_FACTOR'] * 100
axes[1].hist(solar_cf, bins=30, color=stack_colors['Solar'], edgecolor='black', alpha=0.7)
axes[1].axvline(solar_cf.median(), color='red', linestyle='--', linewidth=2, label=f'Median: {solar_cf.median():.1f}%')
axes[1].set_xlabel('Capacity Factor (%)')
axes[1].set_ylabel('Number of solar farms')
axes[1].set_title('Solar Farm Capacity Factor Distribution — 2025')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nWind: {len(wind_cf)} farms, median CF = {wind_cf.median():.1f}%, range = {wind_cf.min():.1f}%–{wind_cf.max():.1f}%")
print(f"Solar: {len(solar_cf)} farms, median CF = {solar_cf.median():.1f}%, range = {solar_cf.min():.1f}%–{solar_cf.max():.1f}%")

In [ ]:
# Regional Generation Mix — Energy share by fuel type
# Shows what fraction of total energy comes from each fuel type in each region
# VIC is coal-heavy, SA is wind-heavy, TAS is hydro-dominated, etc.
# This is the "generation mix" metric you see in AEMO reports and energy dashboards

regional_mix = (dispatch_enriched
    .groupby(['REGIONID', 'FUEL_TYPE'])['DISPATCHLOAD']
    .sum()
    .unstack(fill_value=0)
)

# Convert to percentage share
regional_mix_pct = regional_mix.div(regional_mix.sum(axis=1), axis=0) * 100

# Stack order for consistent colour mapping
stack_order_present = [f for f in stack_order if f in regional_mix_pct.columns]

fig, ax = plt.subplots(figsize=(10, 6))
regional_mix_pct[stack_order_present].plot.bar(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    width=0.7
)

ax.set_xlabel('Region')
ax.set_ylabel('Energy share (%)')
ax.set_title('NEM Generation Mix by Region — 2025')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(regional_mix_pct.index, rotation=0)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

print("\nRegional generation mix (% of total energy):")
print(regional_mix_pct.round(1))

In [ ]:
# Top 20 Generators by Energy Output
# Who are the workhorse units in the NEM?
# Baseload coal units dominate this list, but you'll also see major wind farms
# and hydro stations. Battery storage won't appear here (small energy, high power).

top_generators = (energy_by_duid
    .sort_values('DISPATCHLOAD', ascending=False)
    .head(20)
    [['DUID', 'REG_CAPACITY', 'DISPATCHLOAD', 'CAPACITY_FACTOR', 'FUEL_TYPE', 'REGIONID']]
)

# Convert dispatch to GWh for readability
top_generators['ENERGY_GWh'] = top_generators['DISPATCHLOAD'] / 12 / 1000  # 5-min MW → GWh

print("Top 20 generators by energy output in 2025:")
print(top_generators[['DUID', 'FUEL_TYPE', 'REGIONID', 'REG_CAPACITY', 'ENERGY_GWh', 'CAPACITY_FACTOR']].to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    range(len(top_generators)),
    top_generators['ENERGY_GWh'],
    color=[stack_colors.get(f, '#B8B8B8') for f in top_generators['FUEL_TYPE']]
)

ax.set_yticks(range(len(top_generators)))
ax.set_yticklabels(top_generators['DUID'], fontsize=9)
ax.set_xlabel('Energy output (GWh)')
ax.set_title('Top 20 NEM Generators by Energy Output — 2025')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly Dispatch Stack — SA
# How does the generation mix evolve month-to-month?
# Winter vs. summer, wind resource seasonality, solar output changes with day length
# SA is an interesting case study — high renewable penetration, no coal baseload

region = 'SA1'

monthly_data = dispatch_enriched[dispatch_enriched['REGIONID'] == region].copy()
monthly_data['month'] = monthly_data['SETTLEMENTDATE'].dt.month

monthly_stack = (monthly_data
    .groupby(['month', 'FUEL_TYPE'])['DISPATCHLOAD']
    .sum()
    .unstack(fill_value=0)
)

# Convert to GWh
monthly_stack = monthly_stack / 12 / 1000

stack_order_present = [f for f in stack_order if f in monthly_stack.columns]

fig, ax = plt.subplots(figsize=(12, 6))
monthly_stack[stack_order_present].plot.bar(
    ax=ax,
    stacked=True,
    color=[stack_colors[f] for f in stack_order_present],
    width=0.8
)

ax.set_xlabel('Month')
ax.set_ylabel('Energy (GWh)')
ax.set_title(f'{region} Monthly Generation Stack — 2025')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
plt.tight_layout()
plt.show()

print("\nMonthly energy by fuel type (GWh):")
print(monthly_stack[stack_order_present].round(0))